In [1]:
import mpmath as mp
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime
import natural_units as nu
from numpy.polynomial.laguerre import laggauss
mp.mp.dps = 25

rho_s     = 1.28e7 * nu.mSun / nu.kpc**3
r_s       = 6.5 * nu.kpc
sigma_fid = 1 / rho_s / r_s
v_fid     = mp.sqrt(4 * mp.pi * nu.G_Newton * rho_s) * r_s
lumi_fid  = mp.power(4 * mp.pi * rho_s * r_s**2, 5/2) * mp.power(nu.G_Newton, 3/2)
t_fid     = 1 / mp.sqrt(4 * mp.pi * nu.G_Newton * rho_s)
C_fid     = mp.power(4 * mp.pi * nu.G_Newton, 3/2) * mp.power(rho_s, 5/2) * r_s**2

t_fid_in_gyr = t_fid / (1e9 * nu.year)

# Model parameters
a = mp.mpf('2.257')
c = mp.mpf('0.6')
my_mass_norm = mp.mpf('0.0') # M_b/(4*pi*rho_s*r_s^3)
my_scale_norm = mp.mpf('0.1') # normalized baryon scale radius, a/r_s
# The following are all the velocity-dependent parameters.
m_chi = 1 * nu.GeV
m_phi = 0.1 * nu.MeV
g_chi = 1e-2
omega = m_phi / m_chi
# omega as a velocity also needs to be converted
my_omega = omega / v_fid
# sigma_0 takes a 1/m to be in the form of sigma/m like SIDM strength
sigma_0 = g_chi**4 / 4 / mp.pi / m_chi**2 / omega**4 / m_chi
my_sigma_0 = sigma_0 / sigma_fid

# We are in place to define particle physics functions.
# differential cross section only takes the dimensionless velocity and angular terms, without the sigma at front.
def diff_cs_ruth(v, w, x): # v for velocity (renormalized), x for cos\theta
    y = v**2 / w**2
    return 1 / 2 / (1 + y * (1 - x) / 2)**2

def diff_cs_moll(v, w, x):
    y = v**2 / w**2
    top  = (3 * x**2 + 1) * y**2 + 4 * y + 4
    down = ( (1 - x**2) * y**2 + 4 * y + 4 )**2
    return top / down

def tot_cs_ruth(v, w): # total cross section but without sigma at front
    v_mp = mp.mpf(v)
    w_mp = mp.mpf(w)
    y = v_mp**2 / w_mp**2
    return 1 / (1+y)

def tot_cs_moll(v, w):
    v_mp = mp.mpf(v)
    w_mp = mp.mpf(w)
    y = v_mp**2 / w_mp**2
    return 1 / (1 + y) - 1 / (y**2 + 2 * y) * mp.log(1 + y)

def I_ruth(v, w): # angular integral of cross section with weight of sin^2(theta)
    v_mp = mp.mpf(v)
    w_mp = mp.mpf(w)
    y = v_mp**2 / w_mp**2
    return 4 * ((2 + y) * mp.log(1 + y) - 2 * y) / y**3

def I_moll(v, w):
    v_mp = mp.mpf(v)
    w_mp = mp.mpf(w)
    y = v_mp**2 / w_mp**2
    top = 2 * (2 * (y**2 + 5 * y + 5) * mp.log(1 + y) - 5 * (y**2 + 2 * y))
    down = y**3 * (2 + y)
    return top / down

def big_int(vd, w, N=40, cs_type="ruth"):
    # Gauss-Laguerre nodes and weights for ∫_0^∞ e^{-x} f(x) dx
    x, wL = laggauss(N)  

    vd_mp = mp.mpf(vd)
    w_mp = mp.mpf(w)

    total = mp.mpf('0.0')
    for xi, wi in zip(x, wL):
        x_mp = mp.mpf(xi)
        wL_mp = mp.mpf(wi)
        v_rel = 2 * vd_mp * mp.sqrt(x_mp)

        if cs_type == "ruth":
            Isig = I_ruth(v_rel, w_mp)
        elif cs_type == "moll":
            Isig = I_moll(v_rel, w_mp)
        else:
            raise ValueError(f"Unknown cs_type '{cs_type}'. Use 'ruth' or 'moll'.")

        f = x_mp**3 * Isig
        total += wL_mp * f

    return 128 * total

In [2]:
my_sigma_0

mpf('30.20213585830301211392735308')